# P7 判定を人に合わせる

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

返信の下書き 80 件（ソラハ電機・架空）に pass / fail を付ける判定役を、作った側のラベルに合わせて直します。🔴 評価者 A・B・C と判定者 4 人は**模擬**（人ではありません）。

- 上から順に実行します。0 のセルは準備です（Colab でも手元の Jupyter でも同じ）。
- API キーが無ければ講師の記録の再生（replay）で、本物の実行と同じ数字が出ます（費用 0）。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る（Colab のときだけ取得とインストールをする）
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/llm-eval-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/llm-eval-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根（common/ がある所）を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/llm.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. AI の呼び先を決める（キーは表示しない）
from common import nb
from common.nb import choose_provider, safe, show, show_file, lab
PROVIDER = choose_provider(IN_COLAB)

## 画面に収める小さな道具
長い行を画面の幅で折り返して表示するだけの関数です（中身は変えません）。

In [ ]:
import io, contextlib
def fit(fn, *a, w=104, **k):   # fn の表示を、全角 2・半角 1 で数えた幅 w で折り返す
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        fn(*a, **k)
    for line in buf.getvalue().splitlines():
        out, n = "", 0
        for ch in line:
            d = 2 if ord(ch) > 0x2E80 else 1
            if n + d > w:
                print(out)
                out, n = "        ", 8
            out, n = out + ch, n + d
        print(out)

## 1. 返信 80 件と、判定させる関数
依頼文は `lab07_judge_calibration/prompts/pointwise_v1.md`・`v2.md`。dev と test に分けてから使います。

In [ ]:
from common import kit, llm as L, judge as J, agreement as A, stats
rc = lab("run_calibrate", "lab07_judge_calibration")
P = "replay" if PROVIDER == "mock" else PROVIDER
S, O = "claude-sonnet-5-5", "claude-opus-5-5"
reps, policy = kit.load_jsonl("replies.jsonl"), kit.load_text("reply_policy.md")
dev, test = rc.split(reps)
def judge_all(prompt, rows, model):
    t = J.load_prompt(f"lab07_judge_calibration/prompts/pointwise_{prompt}.md")
    return [J.pointwise(t, inquiry=r["inquiry"], reply=r["reply"], policy=policy, received=r["received"],
                        provider=P, model=model)[0] for r in rows]
print(f"返信 {len(reps)} 件 → dev {len(dev)} / test {len(test)}")

## 2. 依頼文 v1 で dev を判定する ― 真陽性率と真陰性率
TPR = 不合格を不合格と見つけた率、TNR = 合格を合格とした率。

In [ ]:
V1, truth = {}, [r["label"] for r in dev]
print(f"dev {len(dev)} 件: 不合格 {truth.count('fail')} / 合格 {truth.count('pass')}")
for model in (S, O):
    with safe():
        pred = V1[model] = judge_all("v1", dev, model)
        m = A.tpr_tnr(pred, truth, positive="fail")
        print(f"{model:<18} TPR {m['tp']}/{m['tp'] + m['fn']} = {kit.pct(m['tpr'])}  TNR {m['tn']}/{m['tn'] + m['fp']} = "
              f"{kit.pct(m['tnr'])}  κ {A.cohen_kappa(pred, truth):.3f}")

## 3. コインを投げても半分は一致する ― 偶然の一致を引いた一致
でたらめに pass / fail を付ける 2 人を 1000 件ずつ作ります。

In [ ]:
import random
rng = random.Random(0)
a = [rng.choice(["pass", "fail"]) for _ in range(1000)]
b = [rng.choice(["pass", "fail"]) for _ in range(1000)]
print(f"コイン 2 枚: 一致 {sum(x == y for x, y in zip(a, b)) / 1000:.3f} / κ {A.cohen_kappa(a, b):+.3f}")
c = ["pass" if rng.random() < .9 else "fail" for _ in range(1000)]
d = ["pass" if rng.random() < .9 else "fail" for _ in range(1000)]
print(f"9 割 pass と言うだけの 2 人: 一致 {sum(x == y for x, y in zip(c, d)) / 1000:.3f} / κ {A.cohen_kappa(c, d):+.3f}")

## 4. 調整用（dev）と試験用（test）に分けるコード
欠陥の種類ごとに半分ずつ。乱数は固定。test は最後まで見ません。

In [ ]:
show(rc.split)

## 5. 分けた結果 ― 欠陥の種類ごとの件数

In [ ]:
from collections import Counter
cd, ct = Counter(r["defect"] or "(欠陥なし)" for r in dev), Counter(r["defect"] or "(欠陥なし)" for r in test)
print(kit.table([[k, cd[k], ct[k]] for k in sorted(cd | ct)], ["欠陥の種類", "dev", "test"]))
print("不合格: dev", sum(r["label"] == "fail" for r in dev), "/ test", sum(r["label"] == "fail" for r in test))

## 6. 依頼文 v1 と v2
v2 は 4 つの点検と返信の決まりを書き足したもの（`lab07_judge_calibration/prompts/`）。

In [ ]:
show_file("lab07_judge_calibration/prompts/pointwise_v1.md", 1, 1)
print()
fit(show_file, "lab07_judge_calibration/prompts/pointwise_v2.md", 1, 9)

## 7. 依頼文 v2 で dev を直し、test で 1 回だけ測る

In [ ]:
V2 = {}
for name, rows in (("dev", dev), ("test", test)):
    truth = [r["label"] for r in rows]
    for model in (S, O):
        with safe():
            pred = V2[(name, model)] = judge_all("v2", rows, model)
            m = A.tpr_tnr(pred, truth, positive="fail")
            print(f"v2 {name:<4} {model:<18} TPR {m['tp']}/{m['tp'] + m['fn']}  TNR {m['tn']}/{m['tn'] + m['fp']}"
                  f"  κ {A.cohen_kappa(pred, truth):.3f}")

## 8. 判定が数えた不合格率を直す（Rogan-Gladen）
校正は dev の TPR / TNR、直すのは test で判定が数えた率。区間はブートストラップ。

In [ ]:
show(A.rogan_gladen)
cal, obs = V2[("dev", S)], V2[("test", S)]
m = A.tpr_tnr(cal, [r["label"] for r in dev], positive="fail")
p_obs = obs.count("fail") / len(obs)
print(f"\n直した率 = ({p_obs:.3f} + {m['tnr']:.3f} - 1) / ({m['tpr']:.4f} + {m['tnr']:.3f} - 1)"
      f" = {A.rogan_gladen(p_obs, m['tpr'], m['tnr']):.3f}")
rg = A.corrected_prevalence_ci(cal, [r["label"] for r in dev], obs, positive="fail", seed=3)
print(f"判定が数えた {kit.pct(rg['observed'])} → 補正 {kit.pct(rg['corrected'])} 95%区間 {kit.ci_str(rg['lo'], rg['hi'])}"
      f"（本当の率 {kit.pct([r['label'] for r in test].count('fail') / len(test))}）")

## 9. 評価者どうしの一致 ― Cohen κ・Fleiss κ・Krippendorff α
🔴 評価者 A・B・C は模擬（乱数のくせで作ったラベル）。人ではありません。

In [ ]:
import itertools
R = ["A", "B", "C"]
lab_ = {k: [x["annotators"][k] for x in reps] for k in R}
rows = []
for a, b in itertools.combinations(R, 2):
    k = A.cohen_kappa(lab_[a], lab_[b])
    rows.append([f"模擬 {a}-{b}", kit.pct(sum(x == y for x, y in zip(lab_[a], lab_[b])) / len(reps)), f"{k:.3f}", A.landis_koch(k)])
print(kit.table(rows, ["組", "一致", "Cohen κ", "目安"]))
units = [[x["annotators"][k] for k in R] for x in reps]
print(f"\n3 人: Fleiss κ = {A.fleiss_kappa(units):.3f} / Krippendorff α = {A.krippendorff_alpha(units):.3f}")
half = [[u[0], u[1], u[2] if i % 2 else None] for i, u in enumerate(units)]
print(f"C が半分しか付けていない場合の α = {A.krippendorff_alpha(half):.3f}（κ では計算できない形）")

## 10. LLM 判定と、模擬の評価者・作った側のラベル
LLM 判定は上の 7 の Sonnet 5.5・依頼文 v2（dev＋test の 80 件）。

In [ ]:
llm = dict(zip([r["id"] for r in dev + test], V2[("dev", S)] + V2[("test", S)]))
pred = [llm[x["id"]] for x in reps]
for k in R:
    print(f"LLM - 模擬 {k}  κ = {A.cohen_kappa(pred, lab_[k]):.3f}   /  作った側のラベル - 模擬 {k}  κ = "
          f"{A.cohen_kappa([x['label'] for x in reps], lab_[k]):.3f}")

## 11. 模擬の判定者とは ― LLM に人物像を与えたもの
🔴 4 人とも Sonnet 5.5 に人物像とガイドラインを渡したもの。人ではありません。

In [ ]:
rp = lab("run_personas", "lab07_judge_calibration")
plist = rp.personas()
for pid, txt in plist.items():
    lines = txt.splitlines()
    print(lines[0].lstrip("# "))
    fit(print, "   " + next(l for l in lines if l.startswith("くせ")))
print()
fit(show_file, "lab07_judge_calibration/prompts/annotate_batch.md", 1, 1)

## 12. ガイドライン v1 と v2
v1 は要約だけの初版。v2 は公開の指針（特定商取引法の返品特約・ISO 10002・QA の評価項目）で書き直した版です。

In [ ]:
fit(show_file, "lab07_judge_calibration/prompts/guideline_v1.md", 1, 9)

## 12-2. ガイドライン v2 の見出し ― 判定の順番・fail にしないもの・迷ったとき・改訂の記録・出典

In [ ]:
fit(show_file, "lab07_judge_calibration/prompts/guideline_v2.md", grep="#")

## 13. 模擬の判定者 4 人が v1 と v2 で 80 件を判定する
10 件ずつ・人ごとに読む順番を変えて判定します（記録の再生）。

In [ ]:
args = kit.parser("demo").parse_args(["--provider", P, "--model", S])
LAB, pids = {}, sorted(plist)
for rnd in ("v1", "v2"):
    with safe():
        LAB[rnd], cost = rp.label_round(reps, rnd, args, plist)
        for name, rows in (("dev", dev), ("test", test), ("全 80", reps)):
            ids = [x["id"] for x in rows]
            fk = A.fleiss_kappa([[LAB[rnd][p][i]["verdict"] for p in pids] for i in ids])
            un = sum(len({LAB[rnd][p][i]["verdict"] for p in pids}) == 1 for i in ids)
            print(f"ガイドライン {rnd}  {name:<5} Fleiss κ {fk:.3f}  全員一致 {un}/{len(ids)}")
        print(f"   費用 {cost:.4f} USD")

## 14. 割れた返信を別のモデルが裁定する
v2 で割れた返信だけを Opus 5.5 が裁定し、「裁定済みの基準」を作ります。

In [ ]:
args.adj_model = O
with safe():
    adj = rp.adjudicate(LAB["v2"], reps, args, "lab07_judge_calibration/out")

## 15. test を裁定済みの基準で測り直す
下がったように見えた κ は、作った側の正解ラベルの誤り（2 件）でした。

In [ ]:
ref = kit.read_json("lab07_judge_calibration/out/adjudicated_reference.json")["reference"]
for model in (S, O):
    pred = V2[("test", model)]
    k0 = A.cohen_kappa(pred, [r["label"] for r in test])
    m = A.tpr_tnr(pred, [ref[r["id"]] for r in test], positive="fail")
    print(f"{model:<18} 作った側のラベルで κ {k0:.3f} → 裁定済みの基準で TPR {m['tp']}/{m['tp'] + m['fn']}"
          f"  TNR {m['tn']}/{m['tn'] + m['fp']}  κ {A.cohen_kappa(pred, [ref[r['id']] for r in test]):.3f}")

## 16. 例題 ― 2 行 2 列の表から κ を手で出す
依頼文 v1・Sonnet 5.5・dev 38 件（上の 2 の本物の判定）。

In [ ]:
pred, truth = V1[S], [r["label"] for r in dev]
m = A.tpr_tnr(pred, truth, positive="fail")
tp, fn, fp, tn, n = m["tp"], m["fn"], m["fp"], m["tn"], len(dev)
print(kit.table([["判定 fail", tp, fp], ["判定 pass", fn, tn]], ["", "正解 fail", "正解 pass"]))
po = (tp + tn) / n
pe = (tp + fp) / n * (tp + fn) / n + (fn + tn) / n * (fp + tn) / n
print(f"\n一致 po = ({tp} + {tn}) / {n} = {po:.3f}")
print(f"偶然の一致 pe = {tp + fp}/{n} × {tp + fn}/{n} + {fn + tn}/{n} × {fp + tn}/{n} = {pe:.3f}")
print(f"κ = (po - pe) / (1 - pe) = {(po - pe) / (1 - pe):.3f}   関数 cohen_kappa {A.cohen_kappa(pred, truth):.3f}")

## 17. 例題 ― ガイドライン v1 で割れた dev の返信を読む
割れたところが、ガイドラインの穴の候補です。

In [ ]:
rows = []
for r in dev:
    vs = [LAB["v1"][p][r["id"]]["verdict"] for p in pids]
    if len(set(vs)) > 1:
        rows.append([r["id"], r["defect"] or "(欠陥なし)", r["label"], " ".join(f"{p[:2]}:{v}" for p, v in zip(pids, vs))])
print(kit.table(rows, ["返信", "作った欠陥", "正解", "模擬の判定者 4 人"]))
print(f"\ndev {len(dev)} 件のうち、割れたのは {len(rows)} 件")

## 18. 割れたところをどう直したか ― v2 の改訂の記録

In [ ]:
fit(show_file, "lab07_judge_calibration/prompts/guideline_v2.md", 48, 55)